# Feature Engineering: Indirect Prompt Injection Detection

Este notebook construye las características para detectar ataques de prompt injection usando clasificadores ML clásicos.

**Objetivo:** Extraer características lingüísticas y embeddings semánticos del dataset analizado en el EDA.

**Estructura:**
1. Load Dataset & EDA Summary
2. Linguistic / Handcrafted Features
3. Semantic Embedding Features
4. Feature Combination & Final Feature Matrix
5. Feature Importance Analysis
6. Save Artifacts

In [3]:
import re
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

---
## SECTION 1 — Load Dataset & EDA Summary

Cargamos el dataset de indirect prompt injection y resumimos los hallazgos clave del EDA.

In [4]:
def resolve_data_path() -> Path:
    candidates = [
        Path("../data/indirect_prompt_injection_bipia_gpt_train.csv"),
        Path("data/indirect_prompt_injection_bipia_gpt_train.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Dataset CSV not found. Expected one of: "
        + ", ".join(str(p) for p in candidates)
    )

DATA_PATH = resolve_data_path()
df = pd.read_csv(DATA_PATH)

print(f"Dataset path: {DATA_PATH}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Dataset path: ../data/indirect_prompt_injection_bipia_gpt_train.csv
Shape: (70000, 4)
Columns: ['context', 'user_intent', 'label', 'source']


In [5]:
# Verificar distribución de clases
print("Class distribution:")
print(df["label"].value_counts().sort_index())
print(f"\nTotal samples: {len(df)}")
print(f"Benign (0): {(df['label'] == 0).sum()} ({(df['label'] == 0).mean()*100:.1f}%)")
print(f"Malicious (1): {(df['label'] == 1).sum()} ({(df['label'] == 1).mean()*100:.1f}%)")

Class distribution:
label
0    35000
1    35000
Name: count, dtype: int64

Total samples: 70000
Benign (0): 35000 (50.0%)
Malicious (1): 35000 (50.0%)


In [6]:
# Verificar valores nulos
print("Null values per column:")
print(df.isna().sum())
assert df.isna().sum().sum() == 0, "Dataset contains null values!"

Null values per column:
context        0
user_intent    0
label          0
source         0
dtype: int64


### Conclusiones del EDA

Del análisis exploratorio previo se identificaron los siguientes hallazgos relevantes para feature engineering:

1. **Balance de clases:** Dataset perfectamente balanceado (35,000 benignos / 35,000 maliciosos)

2. **Confounding variable:** `label` está alineado 1:1 con `source` (BIPIA → malicious, GPT-4o-mini → benign). Esto implica que el modelo podría aprender patrones de estilo/origen en lugar de semántica de inyección.

3. **Longitudes discriminativas:**
   - `context`: mediana 961 chars, p95 = 3,305 chars
   - `user_intent`: mediana 56 chars, p95 = 343 chars
   - Los ejemplos maliciosos tienden a tener contextos con instrucciones embebidas

4. **Tipos de contenido detectados:**
   - Tablas Markdown: ~28.9%
   - Code fences: ~19.7%
   - Formato email: ~11.3%
   - URLs: ~8.9% (más frecuente en maliciosos: 14.8% vs 3%)
   - SQL keywords: ~7.9% (más en maliciosos: 10.1% vs 5.6%)

5. **Patrones de inyección:**
   - Frases directas como "ignore previous" no aparecen con alta frecuencia
   - "do not" aparece más en maliciosos (10.7% vs 3.9%)
   - Las inyecciones son sutiles y embebidas en contenido externo

**Implicación para features:** Necesitamos características que capturen tanto patrones léxicos explícitos como estructura semántica subyacente.

In [7]:
# Preparar columna de texto para feature extraction
# Usamos 'context' ya que es donde se embeben las instrucciones maliciosas
df["text"] = df["context"].astype(str)
y = df["label"].values

print(f"Text column created from 'context'")
print(f"Label array shape: {y.shape}")

Text column created from 'context'
Label array shape: (70000,)
